Groverin algoritmi. Ks. Wikipedia miten toimii. Tässä implementaatiossa käytetty oraakkelissa Z-gatea, voi käyttää myös CNOT tai X.

In [ ]:
from qiskit.quantum_info import Statevector
from qiskit import transpile, QuantumCircuit
#from qiskit.circuit import Parameter
from qiskit_aer import AerSimulator
from qiskit.circuit.library import ZGate
from qiskit.visualization import plot_histogram
import random
import numpy as np

N = 16

# On olemassa mcx mutta ei mcz... bro what
# Tehdään siis oma mcz:

mass_z = ZGate().control(num_ctrl_qubits=N-1)

def simulate_measurements(qc_samples, shots):
    backend = AerSimulator(seed_simulator = random.randint(0,10000000))
    qc_t = transpile(qc_samples, backend)
    job = backend.run(qc_t, shots=shots)
    return job.result()
# Applyataan suoraan olemassa olevaan circuittiin
def grover_oracle(target: str, N: int):
    N = len(target)
    oracle = QuantumCircuit(N, name="O")
    for i, bit in enumerate(reversed(target)):
        if bit == "0":
            oracle.x(i)
    oracle.append(mass_z, list(range(N)))

    for i, bit in enumerate(reversed(target)):
        if bit == "0":
            oracle.x(i)
    
    return oracle.to_gate()

def grover_diffuser(N: int):
    diffuser = QuantumCircuit(N, name="D")
    diffuser.h(range(N)); diffuser.x(range(N))
#    qc.mcz(range(1, n), 0) # control, target
    diffuser.append(mass_z, list(range(1, N))+[0])
    diffuser.x(range(N)); diffuser.h(range(N))
    return diffuser.to_gate()


In [ ]:
# Alustetaan n-tiloilla
target = format(random.getrandbits(N), f'0{N}b')
print(list(range(1,5)))
print(target)
qc = QuantumCircuit(N)
# Kaikkiin Hadamard:
qc.h(range(N))
init_state = Statevector.from_instruction(qc)
print(init_state)
iterations = int(np.floor((np.pi/4) * np.sqrt(2**N)))
oracle_gate = grover_oracle(target, N=N)
diffuser_gate = grover_diffuser(N)

for _ in range(iterations):
    qc.append(oracle_gate, list(range(N)))
    qc.append(diffuser_gate, list(range(N)))
qc_samples = qc.copy()
qc_samples.measure_all()
grover_result = simulate_measurements(qc_samples, 1000000)
#plot_histogram(grover_result.get_counts())
print(grover_result.get_counts())

#qc.draw("text")